# 06 — Analyse d'erreurs et recommandations

**Projet** : Prévision de consommation électrique multi-horizons avec scikit-learn
**Principe** : un MAPE global ne dit **jamais** quoi corriger. En prévision, 4 % d'erreur moyenne
peuvent cacher 15 % sur les vagues de froid et 2 % le reste du temps — et ce sont les 15 % qui
coûtent cher, parce que ce sont les jours où l'écart au prévu se paie au prix spot.

Ce notebook descend au niveau de la ligne, dans quatre directions : **par horizon** (quatre produits
différents), **par régime** (là où se perd l'argent), **par saison** (un biais saisonnier ne se
corrige pas avec des arbres), et **dans le temps** (résidus autocorrélés = structure non apprise).
Il mesure enfin la couverture réelle des intervalles et teste le recalibrage adaptatif que le
rapport recommande.

Le split de **test** n'est utilisé qu'ici — une seule fois.

## Objectifs pédagogiques

1. Produire l'évaluation complète avec l'objet de production (`Evaluator`), y compris les intervalles et le backtest.
1. Ventiler l'erreur **par horizon, par régime, par mois** : trois lectures qui appellent trois correctifs différents.
1. Lire les **pires journées** une par une, et distinguer l'erreur isolée de l'erreur systématique.
1. Mesurer l'**autocorrélation des résidus** : ce qui reste de structure temporelle est de l'information non apprise, pas du bruit.
1. Vérifier la **couverture réelle des intervalles** et tester le recalibrage adaptatif, en le ventilant entre jours calmes et régimes extrêmes.
1. Formuler des recommandations **chiffrées**, chacune rattachée à une mesure.

**Objectifs transverses du dépôt**

- Construire un jeu supervisé par expansion temporelle (origine x horizon) et formaliser le contrat d'antériorité de chaque feature : connue à l'origine, connue par avance, ou interdite.
- Comprendre pourquoi un split chronologique s'impose et ce que coûte concrètement une validation croisée aléatoire sur une série temporelle.
- Comparer un modèle appris à trois références triviales (persistance, naif saisonnier, moyenne glissante) et quantifier la valeur ajoutée réelle plutôt que le R².

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous les
# lignes INFO de production. Les avertissements réels restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (4800 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 4800

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 4.0)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
# --- Contrat de prévision : tout est lu dans la configuration, rien n'est codé en dur -----------
FORECAST_CONF = dict(CONFIG.model_dump().get("load_forecasting") or {})
HORIZON_COLUMN = str(FORECAST_CONF.get("horizon_column") or "horizon_days")
HORIZONS = tuple(int(value) for value in (FORECAST_CONF.get("horizons") or ()))
LONG_HORIZON = int(FORECAST_CONF.get("long_horizon") or (max(HORIZONS) if HORIZONS else 7))
INTERVAL_LEVEL = float(FORECAST_CONF.get("interval_level") or 0.90)
INTERVAL_METHOD = str(FORECAST_CONF.get("interval_method") or "normalized_conformal")
INTERVAL_SCALE = str(FORECAST_CONF.get("interval_scale_column") or "load_last_observed")
BACKTEST_FOLDS = int(FORECAST_CONF.get("backtest_folds") or 5)

TARGET = str(CONFIG.data.target)
TIME_COLUMN = str(CONFIG.data.time_column or "origin_date")
TARGET_DATE = "target_date" if "target_date" in raw.columns else TIME_COLUMN
EVENT_COLUMN = "event_type" if "event_type" in raw.columns else None
NAIVE_COLUMN = "load_seasonal_naive" if "load_seasonal_naive" in raw.columns else None
PERSIST_COLUMN = "load_last_observed" if "load_last_observed" in raw.columns else None
TEMP_FORECAST = "temperature_forecast_c" if "temperature_forecast_c" in raw.columns else None


def mape(truth: Any, predicted: Any) -> float:
    """Mean absolute percentage error, in percent, ignoring zero denominators.

    Args:
        truth: Observed values.
        predicted: Forecast values.

    Returns:
        The MAPE in percent (``nan`` when nothing is measurable).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & (np.abs(observed) > 1e-8)
    if not usable.any():
        return float("nan")
    return float(np.mean(np.abs((observed[usable] - forecast[usable]) / observed[usable])) * 100.0)


def mase(truth: Any, predicted: Any, reference: Any) -> float:
    """Mean absolute scaled error: model error over the naive reference error.

    Args:
        truth: Observed values.
        predicted: Forecast values.
        reference: Naive reference forecast on the same rows.

    Returns:
        The MASE (below 1 means better than the reference).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    naive = np.asarray(reference, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & np.isfinite(naive)
    scale = float(np.mean(np.abs(observed[usable] - naive[usable])))
    if not usable.any() or scale < 1e-9:
        return float("nan")
    return float(np.mean(np.abs(observed[usable] - forecast[usable])) / scale)


def daily_series(frame: pd.DataFrame) -> pd.DataFrame:
    """Collapse the (origin, horizon) panel into one row per target day.

    Le panel contient plusieurs lignes par jour cible (une par horizon) qui portent **la même**
    consommation : la série quotidienne se reconstruit en dédupliquant sur la date cible.

    Args:
        frame: Raw panel.

    Returns:
        One row per target day, sorted chronologically.
    """
    unique = frame.drop_duplicates(subset=[TARGET_DATE]).copy()
    unique[TARGET_DATE] = pd.to_datetime(unique[TARGET_DATE])
    return unique.sort_values(TARGET_DATE).reset_index(drop=True)


print(
    f"contrat de prévision : horizons={HORIZONS} colonne='{HORIZON_COLUMN}' "
    f"intervalle={INTERVAL_METHOD} (niveau {INTERVAL_LEVEL:.0%}, échelle '{INTERVAL_SCALE}')"
)
print(
    f"cible='{TARGET}' origine='{TIME_COLUMN}' cible_date='{TARGET_DATE}' "
    f"régimes='{EVENT_COLUMN}' naif='{NAIVE_COLUMN}'"
)

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    callbacks=[],
)
print(MODEL.summary())

## 1. Évaluation complète sur le split de test

In [ ]:
# Évaluation complète sur le split de TEST — utilisé une seule fois, ici.
from src.evaluation.evaluator import Evaluator

EVALUATOR = Evaluator.from_config(MODEL, CONFIG.model_dump(), NB_PATHS)
RESULT = EVALUATOR.evaluate(
    PREPARED["X_test"],
    PREPARED["y_test"],
    split="test",
    context=PREPARED["enriched"]["test"],
)

metrics_frame = pd.DataFrame(
    {
        "métrique": list(RESULT.metrics),
        "valeur": [round(float(RESULT.metrics[name]), 4) for name in RESULT.metrics],
    }
)
print(f"lignes évaluées        : {RESULT.n_samples}")
print(f"métrique primaire      : {RESULT.primary_metric} = {RESULT.primary_value:.4f}")
print(f"biais moyen            : {RESULT.bias:+.2f} %")
print(f"bande de tolérance ± {EVALUATOR.tolerance_pct:.0f} % : {RESULT.coverage:.1%} des lignes")
print(f"couverture d'intervalle: {RESULT.interval_coverage:.1%} (nominal {INTERVAL_LEVEL:.0%})")
metrics_frame

**Ce qu'il faut retenir**

- La métrique de décision est `mape` = **__PRIMARY_VALUE__**. Elle se lit avec le gain sur le naif saisonnier : sans cette comparaison, un MAPE de 4 % est impossible à qualifier.
- Deux « couvertures » cohabitent et ne mesurent pas la même chose : la **bande de tolérance** (part des prévisions à moins de ±5 % du réalisé, lecture métier) et la **couverture d'intervalle** (part des réalisations dans l'intervalle publié, lecture statistique). Les confondre est une erreur classique de revue.
- Le `context` passé à `evaluate()` apporte les colonnes brutes (régime, date cible, échelle de dispersion) : sans lui, la ventilation par régime serait impossible.

## 2. Erreur par horizon : quatre produits, pas un

In [ ]:
# Erreur par horizon : quatre produits différents, pas un.
from src.visualization.plots import ForecastingPlots

PLOTS = ForecastingPlots(NB_PATHS.figures_dir)
table = RESULT.per_horizon
columns = [
    column
    for column in [
        "horizon",
        "rows",
        "mape_pct",
        "smape_pct",
        "mae_mw",
        "rmse_mw",
        "bias_pct",
        "naive_mape_pct",
        "improvement_vs_naive_pct",
        "interval_coverage_pct",
    ]
    if column in table.columns
]
print(table[columns].round(3).to_string(index=False))
print()
for name in ("forecast_vs_actual", "error_by_horizon", "improvement_vs_baselines"):
    path = getattr(PLOTS, name)(RESULT)
    if path is not None:
        display(Image(path, width=620))

**Ce qu'il faut retenir**

- L'erreur croît avec l'horizon pour **deux** raisons distinctes : la prévision météo se dégrade, et l'information de court terme (dernière valeur connue) perd en pertinence. Les séparer demande de regarder la colonne de prévision de température, pas seulement le MAPE.
- Le gain sur le naif **décroît** avec l'horizon : c'est attendu, le naif saisonnier est d'autant plus fort que la cible est proche d'une semaine déjà observée.
- Publier un seul chiffre pour tous les horizons reviendrait à garantir le J+7 au prix du J+1. L'erreur attendue est donc publiée **par horizon** avec chaque prévision.

## 3. Erreur par régime : là où se perd l'argent

In [ ]:
# Erreur par régime : là où se perd l'argent.
predictions = RESULT.predictions
if EVENT_COLUMN and EVENT_COLUMN in predictions.columns:
    truth_safe = predictions["y_true"].where(predictions["y_true"].abs() > 1e-9)
    regime = (
        predictions.assign(biais_pct=100.0 * predictions["residual"] / truth_safe)
        .groupby(EVENT_COLUMN)
        .agg(
            lignes=("ape_pct", "size"),
            mape_pct=("ape_pct", "mean"),
            p95_pct=("ape_pct", lambda values: float(np.percentile(values, 95))),
            biais_pct=("biais_pct", "mean"),
            erreur_totale_mw=("absolute_error", "sum"),
        )
        .sort_values("mape_pct", ascending=False)
    )
    regime["part_des_lignes_pct"] = (100.0 * regime["lignes"] / regime["lignes"].sum()).round(2)
    regime["part_de_l_erreur_pct"] = (
        100.0 * regime["erreur_totale_mw"] / regime["erreur_totale_mw"].sum()
    ).round(2)
    print(regime.round(2).to_string())
    print()
    path = PLOTS.error_by_regime(RESULT)
    if path is not None:
        display(Image(path, width=620))
else:
    print("aucune colonne de régime dans le contexte évalué")

**Ce qu'il faut retenir**

- La colonne « part de l'erreur » est celle qui décide : un régime à 4 % des lignes mais 20 % de l'erreur totale est une priorité, alors qu'un régime à 20 % des lignes et 10 % de l'erreur ne l'est pas.
- Le **biais** par régime est plus parlant que le MAPE : une sous-prévision systématique en vague de froid fait acheter en urgence au prix spot, une sur-prévision fait sur-engager de la production. Les deux coûtent, pas de la même façon.
- Un régime jamais vu en entraînement ne se corrige pas par plus d'arbres : il se corrige par une variable de régime, ou se signale par un indicateur de confiance qui déclenche une revue humaine.

## 4. Erreur par saison : le biais qui se corrige autrement

In [ ]:
# Erreur par mois : un biais saisonnier se corrige par une variable de régime, pas par des arbres.
if "target_month" in predictions.columns:
    month_column = "target_month"
elif TIME_COLUMN in predictions.columns:
    month_column = pd.to_datetime(predictions[TIME_COLUMN]).dt.month
else:
    month_column = None

if month_column is not None:
    months = predictions[month_column] if isinstance(month_column, str) else month_column
    season = (
        predictions.assign(mois=months)
        .groupby("mois")
        .agg(
            lignes=("ape_pct", "size"),
            mape_pct=("ape_pct", "mean"),
            biais_pct=("relative_error_pct", "mean"),
        )
        .sort_values("mape_pct", ascending=False)
    )
    print(season.round(2).to_string())
    print()
    worst = season.index[0]
    print(
        f"pire mois : {worst} (MAPE {season['mape_pct'].iloc[0]:.2f} %, "
        f"biais {season['biais_pct'].iloc[0]:+.2f} %)"
    )
    path = PLOTS.bias_by_month(RESULT)
    if path is not None:
        display(Image(path, width=620))

**Ce qu'il faut retenir**

- Un biais concentré sur quelques mois est presque toujours un effet saisonnier mal appris (thermo-sensibilité sous-estimée en hiver, effet vacances sur-estimé en été).
- Le correctif n'est pas plus de complexité : c'est une variable de régime, un recalibrage saisonnier, ou une feature de profil par mois — toutes choses que le constructeur de features déclaratif sait produire.
- Un biais **stable** sur tous les mois est un autre problème : il se corrige par la perte d'entraînement ou par un terme correctif global.

## 5. Les pires journées, lues une par une

In [ ]:
# Les pires journées : à lire une par une avant de conclure.
errors = RESULT.errors
columns = [
    column
    for column in [
        "sample_id",
        TIME_COLUMN,
        TARGET_DATE,
        HORIZON_COLUMN,
        EVENT_COLUMN,
        "y_true",
        "y_pred",
        "ape_pct",
        "residual",
    ]
    if column and column in errors.columns
]
print(errors[columns].head(12).round(2).to_string(index=False))
print()
path = PLOTS.worst_errors(RESULT)
if path is not None:
    display(Image(path, width=620))
print()
# Une erreur isolée sur un arrêt industriel inédit n'appelle pas le même correctif qu'une erreur
# systématique sur tous les lundis de vacances scolaires : on distingue les deux par la répétition.
if EVENT_COLUMN and EVENT_COLUMN in errors.columns:
    repetition = errors.head(50)[EVENT_COLUMN].value_counts()
    print("composition des 50 pires lignes par régime :")
    print(repetition.to_string())

**Ce qu'il faut retenir**

- Une erreur isolée sur un arrêt industriel inédit n'appelle pas le même correctif qu'une erreur systématique sur tous les lundis de vacances scolaires : la composition des 50 pires lignes par régime tranche la question.
- Le P95 d'erreur se lit avec le MAPE : un P95 très au-dessus du MAPE moyen signale une queue lourde, donc un risque opérationnel que la moyenne cache.
- Ces journées sont candidates à la revue humaine **avant** publication : le prédicteur produit déjà l'indicateur de confiance qui les signale.

## 6. Autocorrélation des résidus : ce qui reste de structure

In [ ]:
# Autocorrélation des résidus : ce que le modèle laisse de structure temporelle.
residual = predictions["residual"].to_numpy(dtype="float64")
ordered = (
    predictions.sort_values(TIME_COLUMN) if TIME_COLUMN in predictions.columns else predictions
)
residual = ordered["residual"].to_numpy(dtype="float64")
n = len(residual)
noise_threshold = 2.0 / np.sqrt(max(n, 1))

rows = []
for lag in (1, 2, 3, 7, 14, 28):
    if lag >= n:
        continue
    coefficient = float(np.corrcoef(residual[lag:], residual[:-lag])[0, 1])
    rows.append(
        {
            "décalage (jours)": lag,
            "autocorrélation": round(coefficient, 3),
            "seuil de bruit": round(noise_threshold, 3),
            "significative": bool(abs(coefficient) > noise_threshold),
            "lecture": (
                "inertie non capturée : un décalage supplémentaire ou un terme AR aiderait"
                if lag == 1 and abs(coefficient) > noise_threshold
                else "saisonnalité hebdomadaire résiduelle : le profil par jour est mal appris"
                if lag == 7 and abs(coefficient) > noise_threshold
                else "structure temporelle résiduelle"
                if abs(coefficient) > noise_threshold
                else "compatible avec du bruit"
            ),
        }
    )
acf_table = pd.DataFrame(rows)
print(acf_table.to_string(index=False))
print()
fig, axis = plt.subplots(figsize=(8, 3.6))
axis.bar(acf_table["décalage (jours)"], acf_table["autocorrélation"], color="#2e75b6")
axis.axhline(noise_threshold, color="#c00000", ls="--", lw=1, label="±2/√n (bruit blanc)")
axis.axhline(-noise_threshold, color="#c00000", ls="--", lw=1)
axis.set_xlabel("décalage (jours)")
axis.set_ylabel("autocorrélation des résidus")
axis.set_title("Un résidu autocorrélé n'est pas du bruit : c'est de l'information non apprise")
axis.legend()
axis.grid(alpha=0.3, axis="y")
plt.show()
print(
    f"\nécart-type des résidus : {residual.std():.1f} | "
    f"erreur relative médiane : {predictions['ape_pct'].median():.2f} % | "
    f"P95 : {predictions['ape_pct'].quantile(0.95):.2f} %"
)

**Ce qu'il faut retenir**

- Un résidu **autocorrélé n'est pas du bruit** : c'est de la dépendance temporelle que le modèle n'a pas capturée. À décalage 1, cela appelle un terme AR ou un décalage supplémentaire ; à décalage 7, un profil hebdomadaire mieux appris.
- Le seuil ±2/√n est la référence de bruit blanc : en dessous, l'autocorrélation observée est compatible avec le hasard, et il ne faut rien corriger.
- Une autocorrélation résiduelle a une seconde conséquence, plus coûteuse : elle invalide l'hypothèse d'indépendance sur laquelle repose un intervalle gaussien. C'est une raison de plus de calibrer les bornes sur des quantiles empiriques.

## 7. Intervalles : couverture réelle, ventilation et recalibrage adaptatif

La couverture nominale est un engagement. Cette section mesure ce qui est **réellement** couvert,
par horizon et par régime, puis exécute le recalibrage quotidien qu'un système réel peut faire
puisque l'erreur d'hier est connue ce matin.

In [ ]:
# Couverture réelle des intervalles, ventilée — puis le recalibrage adaptatif recommandé.
table = RESULT.intervals
columns = [
    column
    for column in [
        "horizon",
        "rows",
        "level_pct",
        "method",
        "scale_column",
        "score_quantile",
        "mean_width_mw",
        "relative_width_pct",
        "coverage_pct",
        "coverage_calm_pct",
        "coverage_extreme_pct",
        "n_extreme",
    ]
    if column in table.columns
]
print(table[columns].round(3).to_string(index=False))
print()
path = PLOTS.interval_coverage(RESULT)
if path is not None:
    display(Image(path, width=620))

# --- Expérience : recalibrage adaptatif ---------------------------------------------------------
# Un système réel connaît chaque matin l'erreur d'hier. On ajoute donc au pool de calibration les
# résidus des lignes **chronologiquement précédentes**, ce que la publication statique ne fait pas.
val_frame = PREPARED["splits"].val
val_forecast = np.asarray(MODEL.predict(PREPARED["X_val"]), dtype="float64").ravel()
val_truth = np.asarray(PREPARED["y_val"], dtype="float64")
if INTERVAL_SCALE in val_frame.columns:
    val_scale = np.abs(val_frame[INTERVAL_SCALE].to_numpy(dtype="float64"))
else:
    val_scale = np.abs(val_forecast)
val_scores = np.abs(val_forecast - val_truth) / np.where(val_scale > 1e-9, val_scale, 1.0)
val_horizons = val_frame[HORIZON_COLUMN].to_numpy()

evaluated = predictions.copy()
if TIME_COLUMN in evaluated.columns:
    evaluated = evaluated.sort_values(TIME_COLUMN).reset_index(drop=True)
scale_eval = (
    np.abs(evaluated[INTERVAL_SCALE].to_numpy(dtype="float64"))
    if INTERVAL_SCALE in evaluated.columns
    else np.abs(evaluated["y_pred"].to_numpy(dtype="float64"))
)
scale_eval = np.where(scale_eval > 1e-9, scale_eval, float(np.median(scale_eval)))
truth_eval = evaluated["y_true"].to_numpy(dtype="float64")
forecast_eval = evaluated["y_pred"].to_numpy(dtype="float64")
residual_eval = evaluated["residual"].to_numpy(dtype="float64")
horizons_eval = evaluated[HORIZON_COLUMN].to_numpy()
calm_eval = (
    (evaluated[EVENT_COLUMN].astype(str) == "none").to_numpy()
    if EVENT_COLUMN and EVENT_COLUMN in evaluated.columns
    else np.ones(len(evaluated), dtype=bool)
)

rows = []
for horizon in sorted(pd.unique(horizons_eval)):
    static_pool = val_scores[val_horizons == horizon]
    static_quantile = float(np.quantile(static_pool, INTERVAL_LEVEL))
    test_mask = horizons_eval == horizon
    static_hit = np.abs(truth_eval[test_mask] - forecast_eval[test_mask]) <= (
        static_quantile * scale_eval[test_mask]
    )
    running = list(static_pool)
    adaptive_hit = np.zeros(int(test_mask.sum()), dtype=bool)
    for position, index in enumerate(np.where(test_mask)[0]):
        current = float(np.quantile(np.asarray(running), INTERVAL_LEVEL))
        adaptive_hit[position] = abs(truth_eval[index] - forecast_eval[index]) <= (
            current * scale_eval[index]
        )
        running.append(abs(residual_eval[index]) / scale_eval[index])
    rows.append(
        {
            "horizon": f"J+{horizon}",
            "lignes": int(test_mask.sum()),
            "quantile statique": round(static_quantile, 4),
            "couverture statique %": round(100.0 * float(static_hit.mean()), 1),
            "couverture adaptative %": round(100.0 * float(adaptive_hit.mean()), 1),
            "gain (points)": round(
                100.0 * (float(adaptive_hit.mean()) - float(static_hit.mean())), 1
            ),
            "couverture adapt. jours calmes %": round(
                100.0 * float(adaptive_hit[calm_eval[test_mask]].mean()), 1
            )
            if calm_eval[test_mask].any()
            else None,
            "couverture adapt. régime extrême %": round(
                100.0 * float(adaptive_hit[~calm_eval[test_mask]].mean()), 1
            )
            if (~calm_eval[test_mask]).any()
            else None,
        }
    )

adaptive_table = pd.DataFrame(rows)
print()
print("=== intervalle conforme normalisé : statique contre recalibrage adaptatif ===")
print(adaptive_table.to_string(index=False))
print(
    f"\ncouverture globale statique   : "
    f"{100.0 * float((np.abs(truth_eval - forecast_eval) <= 0).mean()):.1f} % (rappel : "
    f"{RESULT.interval_coverage:.1%} mesurée par l'évaluateur)"
)
GLOBAL_STATIC = float(np.mean([row["couverture statique %"] for row in rows]))
GLOBAL_ADAPTIVE = float(np.mean([row["couverture adaptative %"] for row in rows]))
print(
    f"moyenne par horizon : statique {GLOBAL_STATIC:.1f} % -> adaptative "
    f"{GLOBAL_ADAPTIVE:.1f} % (gain {GLOBAL_ADAPTIVE - GLOBAL_STATIC:+.1f} point)"
)

**Ce qu'il faut retenir**

- La couverture globale est la moyenne de deux populations très différentes : les jours calmes, proches du nominal, et les jours de régime exceptionnel, où l'intervalle ne protège **rien**. Publier le seul chiffre global serait trompeur.
- Ce n'est pas un défaut de calibration mais sa limite structurelle : la fenêtre de validation ne contient pas le régime, donc aucun quantile appris sur elle ne peut le couvrir. Élargir la fenêtre pour couvrir toutes les saisons est le correctif de fond.
- La normalisation par le niveau (`INTERVAL_SCALE`) est ce qui rend l'intervalle utilisable d'une saison à l'autre : le bruit de la série est multiplicatif, donc une largeur absolue calibrée en été est deux fois trop étroite en hiver.
- Le recalibrage adaptatif gagne quelques points sans élargir davantage l'intervalle, parce qu'il suit la dispersion **récente** au lieu de la dispersion moyenne de la fenêtre de calibration. Son coût est nul : il réutilise les résidus déjà publiés.
- Il ne corrige toutefois pas les régimes inédits — c'est le rôle de l'indicateur de confiance et de la revue humaine, pas de l'intervalle.

## 8. Facteurs contributifs

In [ ]:
# Facteurs contributifs : quelles variables portent la prévision.
importance = RESULT.feature_importance
if importance is None or getattr(importance, "empty", True):
    print("le modèle n'expose pas d'importance : section sans objet")
else:
    frame = importance.copy()
    name_column = "feature" if "feature" in frame.columns else frame.columns[0]
    value_column = "importance" if "importance" in frame.columns else frame.columns[-1]
    frame["famille"] = np.select(
        [
            frame[name_column].astype(str).str.contains("temp|hdd|cdd", regex=True),
            frame[name_column].astype(str).str.startswith("target_"),
            frame[name_column].astype(str).str.startswith("load_"),
        ],
        ["météo", "calendrier", "historique"],
        default="autre",
    )
    print(frame.sort_values(value_column, ascending=False).head(15).round(4).to_string(index=False))
    print()
    print("poids par famille :")
    print(
        frame.groupby("famille")[value_column]
        .sum()
        .sort_values(ascending=False)
        .round(3)
        .to_string()
    )
    path = PLOTS.feature_importance(RESULT)
    if path is not None:
        display(Image(path, width=620))

**Ce qu'il faut retenir**

- L'importance par permutation dit **quelle** variable compte, pas dans quel sens ni en quelle unité : pour la thermo-sensibilité en MW/°C, il faut repasser par le modèle linéaire du notebook 04.
- Le regroupement par famille (météo / calendrier / historique) est plus utile que le classement individuel : il dit quelle **source** de données mérite un investissement (un meilleur contrat météo, un calendrier plus fin).
- Une variable dominante qui serait une métadonnée du jour cible signerait une fuite : c'est le dernier contrôle du contrat d'antériorité.

## 9. Diagnostics consolidés

In [ ]:
# Synthèse des diagnostics produits par l'évaluateur.
extras = dict(RESULT.extras or {})
keys = [
    "naive_mape",
    "improvement_vs_naive_pct",
    "residual_std",
    "bias_pct",
    "median_ape_pct",
    "p95_ape_pct",
    "worst_month_bias_pct",
    "interval_method",
    "interval_scale_column",
    "interval_source",
]
print("=== diagnostics ===")
for key in keys:
    if key in extras:
        value = extras[key]
        print(
            f"  {key:28s} = {value:.4f}"
            if isinstance(value, (int, float))
            else f"  {key:28s} = {value}"
        )

backtest = RESULT.backtest
if backtest is not None and not backtest.empty:
    print()
    print("=== backtest à origine glissante (modèle figé) ===")
    print(backtest.round(3).to_string(index=False))
    path = PLOTS.backtest_stability(RESULT)
    if path is not None:
        display(Image(path, width=620))

segments = RESULT.per_segment
if segments is not None and not segments.empty:
    print()
    print("=== erreur par segment déclaré ===")
    print(segments.round(3).head(20).to_string(index=False))

**Ce qu'il faut retenir**

- Le backtest à modèle figé mesure la **stabilité temporelle**, pas la performance : une dérive croissante du premier au dernier repli annonce un modèle qui vieillit.
- La dispersion entre replis est un critère de succès à part entière — un modèle instable coûte plus cher à exploiter qu'un modèle légèrement moins précis et stable.
- Les segments déclarés dans la configuration donnent une lecture complémentaire (niveau de consommation, jour de la semaine) sans avoir à réécrire le notebook.

## 10. Recommandations

Les recommandations ci-dessous sont produites par le générateur de rapport à partir des mesures de
ce notebook. Celles déclarées dans le manifeste, à titre de référence métier :

1. Toujours comparer le modèle à trois références triviales (persistance, naif saisonnier, climatologie) : battre un apprentissage de référence n'a aucune valeur opérationnelle.
2. Valider par **origine glissante** (rolling-origin backtest) et jamais par validation croisée aléatoire : le découpage chronologique du split principal ne suffit pas à détecter une fuite de features.
3. Publier une prévision par horizon et un intervalle, jamais un chiffre unique : l'acheteur d'énergie raisonne en risque, pas en point.
4. Piloter sur le MAPE et le MASE plutôt que sur la RMSE seule : la RMSE est dominée par les pointes hivernales et fait croire à une dégradation quand seule l'amplitude saisonnière augmente.
5. Surveiller la thermo-sensibilité apprise (MW par degré) : elle dérive avec la rénovation du parc et l'électrification des usages, et c'est le premier signe d'un modèle périmé.
6. Ne jamais rogner les extrêmes au pré-traitement : la vague de froid est exactement le jour où l'erreur coûte le plus cher. Préférer une feature de régime et un indicateur de confiance.
7. Documenter le contrat d'antériorité de chaque feature (connue à l'origine / connue par avance / interdite) : c'est la revue qui évite 90 % des fuites en séries temporelles.

In [ ]:
# Recommandations : chacune est rattachée à une mesure de ce notebook, pas à une opinion.
from src.evaluation.reports import DEFAULT_THRESHOLDS

EXTRAS = dict(RESULT.extras or {})
BIAS_PCT = float(EXTRAS.get("bias_pct", float("nan")))
GAIN_PCT = float(EXTRAS.get("improvement_vs_naive_pct", float("nan")))
horizon_table = RESULT.per_horizon
long_horizon = (
    horizon_table[horizon_table["horizon"] == LONG_HORIZON]["mape_pct"].iloc[0]
    if "horizon" in horizon_table.columns and (horizon_table["horizon"] == LONG_HORIZON).any()
    else float("nan")
)
print("=== état mesuré ===")
print(
    f"  MAPE global                : {RESULT.metrics.get('mape', float('nan')):.3f} % "
    f"(seuil {DEFAULT_THRESHOLDS['mape_max']:.1f} %)"
)
print(
    f"  MAPE à J+{LONG_HORIZON}                 : {long_horizon:.3f} % "
    f"(seuil {DEFAULT_THRESHOLDS['mape_long_horizon_max']:.1f} %)"
)
print(
    f"  gain sur le naif saisonnier: {GAIN_PCT:.1f} % "
    f"(seuil {DEFAULT_THRESHOLDS['improvement_vs_naive_min']:.0f} %)"
)

print(
    f"  couverture d'intervalle    : {100.0 * RESULT.interval_coverage:.1f} % "
    f"(nominal {INTERVAL_LEVEL:.0%}, seuil "
    f"{DEFAULT_THRESHOLDS['interval_coverage_min']:.0f} %)"
)
print(
    f"  biais moyen                : {BIAS_PCT:+.2f} % "
    f"(seuil ±{DEFAULT_THRESHOLDS['bias_max']:.1f} %)"
)
print(
    f"  recalibrage adaptatif      : {GLOBAL_STATIC:.1f} % -> {GLOBAL_ADAPTIVE:.1f} % "
    f"({GLOBAL_ADAPTIVE - GLOBAL_STATIC:+.1f} point)"
)
print()
print("=== plan d'action, par ordre de rapport gain / coût ===")
from src.evaluation.reports import ReportBuilder  # noqa: E402

REPORTER = ReportBuilder(NB_PATHS, config=CONFIG.model_dump())
for index, recommendation in enumerate(REPORTER.recommendations(RESULT), start=1):
    print(f"{index:2d}. {recommendation}")

## 11. Rapport exécutable

In [ ]:
# Le rapport généré par le pipeline : mêmes chiffres, mêmes seuils, régénérable à chaque run.
from src.evaluation.reports import ReportBuilder

REPORTER = ReportBuilder(NB_PATHS, config=CONFIG.model_dump())
WRITTEN = REPORTER.build(RESULT, model=MODEL)
print(f"{len(WRITTEN)} artefacts écrits dans {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")
for name, artefact in sorted(WRITTEN.items()):
    print(f"  {name:22s} {Path(artefact).name}")
print()
Markdown(Path(WRITTEN["report"]).read_text(encoding="utf-8")[:3000] + "\n\n[…]")

**Ce qu'il faut retenir**

- Le rapport mélange **chiffres calculés** et **verdict sur les seuils déclarés** : il est régénérable à chaque run et ne dépend d'aucune saisie manuelle.
- Le même contenu est écrit en Markdown (lecture), en JSON (CI, dashboard) et en CSV (tableurs, revue métier) : trois consommateurs, une seule source de vérité.
- Ce qui reste ouvert, assumé : une seule région, une prévision météo sans scénarios P10/P50/P90, et un backtest à modèle figé en routine. Chacun de ces points est un chantier identifié, pas un angle mort.